<a href="https://colab.research.google.com/github/marcelalozano27-ship-it/machine-learning-2026-spring/blob/main/CA-05/CA_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##kNN - K Nearest Neighbors (CA-05)
### Marcela Lozano and Brandon Richard

The following is a movie recommendation system using K Nearest Neighbors. The model is able to compare a target movie profile to all movies in a given dataset and will return k of the most similar movies based on the distances between features

##### Import the necessary libraries for the movie recommendation model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

##### Read the data into a dataframe directly from the github link provided

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/ArinB/MSBA-CA-Data/main/CA05/movies_recommendation_data.csv')
display(df.head())

,Movie ID,Movie Name,IMDB Rating,Biography,Drama,Thriller,Comedy,Crime,Mystery,History,Label
0,58,The Imitation Game,8.0,1,1,1,0,0,0,0,0
1,8,Ex Machina,7.7,0,1,0,0,0,1,0,0
2,46,A Beautiful Mind,8.2,1,1,0,0,0,0,0,0
3,62,Good Will Hunting,8.3,0,1,0,0,0,0,0,0
4,97,Forrest Gump,8.8,0,1,0,0,0,0,0,0


In [ ]:
df['IMDB Rating'].describe()

,IMDB Rating
count,30.000000
mean,7.696667
std,0.666169
min,5.900000
25%,7.300000
50%,7.750000
75%,8.175000
max,8.800000


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Movie ID     30 non-null     int64  
 1   Movie Name   30 non-null     object 
 2   IMDB Rating  30 non-null     float64
 3   Biography    30 non-null     int64  
 4   Drama        30 non-null     int64  
 5   Thriller     30 non-null     int64  
 6   Comedy       30 non-null     int64  
 7   Crime        30 non-null     int64  
 8   Mystery      30 non-null     int64  
 9   History      30 non-null     int64  
 10  Label        30 non-null     int64  
dtypes: float64(1), int64(9), object(1)
memory usage: 2.7+ KB


In [ ]:
print("Shape of dataset:", df.shape)
print("\nColumn names:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

Shape of dataset: (30, 11)

Column names:
Index(['Movie ID', 'Movie Name', 'IMDB Rating', 'Biography', 'Drama',
       'Thriller', 'Comedy', 'Crime', 'Mystery', 'History', 'Label'],
      dtype='object')

Data Types:
Movie ID         int64
Movie Name      object
IMDB Rating    float64
Biography        int64
Drama            int64
Thriller         int64
Comedy           int64
Crime            int64
Mystery          int64
History          int64
Label            int64
dtype: object


##### Since KNN uses numerical distance calculations, we created a dataframe with only the numeric columns. For the recommender model we do not need features such as Movie ID or Name since they are identifiers and not predictive features.

In [ ]:
features=df.drop(columns=['Movie ID', "Movie Name", "Label"])

###Feature Scaling
##### Scaling was necessary because KNN is dependent on distance calculations. The features have to be on comparable scales to avoid any variables influencing the distance more than others. By standardizing the inputs we make sure the contribution of each feature is balanced.

In [ ]:
scaler=StandardScaler()
features_scaled=scaler.fit_transform(features)

In [ ]:
knn=NearestNeighbors(n_neighbors=5, metric='euclidean')
knn.fit(features_scaled)

NearestNeighbors(metric='euclidean')

###Nearest Neighbors Model
The NearestNeighbors model is used to find movies most similar to a given input representing a movie profile. For this specific case we configured the model to return 5 of the nearest neighbors, most similar movies, using Euclidean distance. This is an example of unsupervised learning because the model is based on similarity rather than prediction of a target variable.

In [ ]:
#Define the characteristics of the input movie we want the recommendations for.
the_post = {
    "IMDB Rating": 7.2,
    "Biography": 1,
    "Drama": 1,
    "Thriller": 0,
    "Comedy": 0,
    "Crime": 0,
    "Mystery": 0,
    "History": 1
}
the_post_df = pd.DataFrame([the_post])
the_post_df = the_post_df[features.columns]

the_post_scaled=scaler.transform(the_post_df)

### Input Movie for Testing
We converted the input array for the feature movie (The Post) into a dataframe for consistency of column names and then used the same scaling approach.

In [ ]:
distances, indices=knn.kneighbors(the_post_scaled)

The kneighbors() function returns the distance between the input movie and the recommended movie as well as the index to map the recommended movies to the original dataset. The closer distances represent the most similar movies.

In [ ]:
recommended_movies=df.iloc[indices[0]]['Movie Name']

print("Top 5 Recommended Movies:")
print(recommended_movies.tolist())

Top 5 Recommended Movies:
['12 Years a Slave', 'Hacksaw Ridge', 'Queen of Katwe', 'The Wind Rises', 'A Beautiful Mind']


In [ ]:
for i, idx in enumerate(indices[0]):
    movie_name=df.iloc[idx]["Movie Name"]
    distance=distances[0][i]
    print(f"{i+1}. {movie_name} (Distance: {distance:.4f})")

1. 12 Years a Slave (Distance: 1.3741)
2. Hacksaw Ridge (Distance: 1.5268)
3. Queen of Katwe (Distance: 3.3473)
4. The Wind Rises (Distance: 3.4569)
5. A Beautiful Mind (Distance: 3.6664)


### Interpretation of Results

The output of the model shows the five movies with the closest distance to the input movie profile with regards to feature similarity. The distances represent how similar each movie is to the target with lower distances indicating proximity to the target and stronger similarity signaling relevant recommendations.